# SDPA 算法规格与 online softmax


## 1. 算法规格

$$
O = \operatorname{softmax}\!\left(\frac{QK^{\mathrm T}}{\sqrt{H}} + M\right)V
$$

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">项</th><th style="text-align:left;">规格（本课程教学版）</th></tr>
<tr><td>Q/K/V</td><td>[B,S,H]，单头；H=64/128，FP16</td></tr>
<tr><td>S</td><td>1024、2048、4096、8192，prefill</td></tr>
<tr><td>mask M</td><td>无 mask（M=0）与 causal 下三角两档；causal 为主线验收项</td></tr>
<tr><td>中间量</td><td>每 batch 的逻辑 logits 为 [S,S]；s/m/l/acc 使用 FP32，分块计算避免存整矩阵</td></tr>
<tr><td>输出</td><td>[B,S,H] FP16</td></tr>
</table>

计算链：QKᵀ（Cube）→ 缩放与 softmax（Vector）→ PV（Cube）。H 始终表示单头的特征维度，不表示 head 数量。


## 2. 朴素三遍法的存储与访存成本

第 5 章 Matmul+Softmax 用了三遍法（一遍最大值、一遍指数和、一遍归一化）。SDPA 的 softmax 作用在 **[S, S] 的行上**：S=4096 时一行 16KB（FP32），三遍法要求 S_mat 完整保存或反复读取——**整张 S_mat 为 64 MiB，远大于片上预算**；一行 16 KiB 是否放得下与同时处理行数及临时空间有关。softmax 后还要乘 V，若 P = softmax(S_mat) 也完整写入 GM，还会增加 P 的写回和读取。三遍法能够实现正确计算；这里要解决的是保存和反复读取中间矩阵的成本。


## 3. online softmax：一遍归约的秘密

把"求最大值、求指数和"合并进**单个遍历**，维护两个运行量：当前最大值 m 与当前指数和 l：

```text
处理新块 s（行内的一段）：
m_new  = max(m,  max(s))          # 更新最大值
l_new  = l * exp(m - m_new) + sum(exp(s - m_new))   # 修正旧贡献 + 累加新块
历史指数贡献的缩放落实到运行量 l/acc，不必保存所有历史 p 块
```

推导（两块情形，可推广到任意块数）：

$$
\sum_j e^{x_j - m_{new}} = e^{m_{old}-m_{new}}\sum_{j\in old} e^{x_j - m_{old}} + \sum_{j\in new} e^{x_j - m_{new}}
$$

最终输出：$O_i = \frac{1}{l_{final}} \sum_j e^{S_{ij} - m_{final}} V_j$——最大值与指数和**在一次遍历中同时收敛**。


## 4. FlashAttention：online softmax × 分块矩阵乘

把 online softmax 与 PV 乘法也分块交织，得到 FlashAttention 的核心循环（对每个 Q 块）：

```text
for 每个 K/V 块 j:
    s_j  = Q_i @ K_j^T / √H          # Cube：小 S 块
    m,l  = online_update(m, l, s_j)   # Vector：行最大值/指数和更新
    p_j  = exp(s_j - m_new)（未归一化，随 m 修正）
    acc  = acc * e^{m_old - m_new} + p_j @ V_j   # Cube：部分和矩阵随 m 修正
O_i = acc / l                          # 最终归一化
```

要点：

1. **acc（[q_block, H] 的 FP32 累加器）随最大值修正**——每个 K 块处理完，历史贡献统一乘缩放因子；
2. S_mat 的分块 [q_block, kv_block] 在片上产生、当块消费，**全程不落 GM**；
3. causal mask 在缩放后的 s_j 上逐块应用（完全可见/完全遮蔽/跨边界三种情形，属主线验收）。


### 手算两块：为什么 l 和 acc 必须一起缩放

取一行已缩放 logits=[0, ln2, ln4]，V=[1,3,5]，按前两项/后一项分块。第一块 m=ln2，p=[1/2,1]，l=3/2，acc=7/2。第二块把 m 提高到 ln4，alpha=1/2，因此 l_new=(3/2)/2+1=7/4，acc_new=(7/2)/2+5=27/4，O=27/7。

直接 softmax 的权重是 [1,2,4]/7，结果同样为 27/7。若只修正 l 而忘记 acc，结果变成 (7/2+5)/(7/4)=34/7；这类错误通常只在后续块出现更大 logits 时暴露。

本节讨论的片上 FlashAttention 路径不落完整 S/P。10.04 的完整教学基线为便于逐段检查，暂用小块 GM workspace 交接：仍不分配 [S,S]，但仍会读写块级 S/P/PV，不能套用片上版的零流量结论。


## 5. 课后练习

1. （推导）手推 online softmax 的 l 更新公式（两块 → 三块归纳），验证 $e^{m_{old}-m_{new}}$ 修正系数的来源。
2. （计算）S=4096、FP32：朴素三遍法的 S_mat GM 流量是多少（写+读×3）？FlashAttention 呢（0 次 S_mat 落 GM）？
3. （思考）acc 为什么要 FP32？用 FP16 累加会有什么风险（提示：大数吃小数）？

In [ ]:
!cat ./answer/10.02_answer.txt

## 6. 本节小结

- 规格：O = softmax(QK^T/√H + M)V，CV 交替流水的教科书算子；
- online softmax：m/l 双运行量一遍收敛，修正系数 $e^{m_{old}-m_{new}}$ 是灵魂；
- Flash 循环 = **分块 QK^T + online 更新 + 修正累加 PV**，S_mat 全程在片上。